<a href="https://colab.research.google.com/github/arturfurlan07-debug/Projeto_Oficina/blob/main/oficina_mecanica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
class No:
    """Nó de lista encadeada: guarda um dado e aponta para o próximo."""
    def __init__(self, dado):
        self.dado = dado
        self.prox = None


class Lista:
    """Lista encadeada simples. adicionar/buscar/remover: O(n)."""
    def __init__(self):
        self.inicio = None
        self.tamanho = 0

    def adicionar(self, dado):  # insere no fim
        novo = No(dado)
        if self.inicio is None:
            self.inicio = novo
        else:
            atual = self.inicio
            while atual.prox is not None:
                atual = atual.prox
            atual.prox = novo
        self.tamanho += 1

    def buscar(self, criterio):  # criterio: função que devolve True/False
        atual = self.inicio
        while atual is not None:
            if criterio(atual.dado):
                return atual.dado
            atual = atual.prox
        return None

    def remover(self, criterio):
        anterior = None
        atual = self.inicio
        while atual is not None:
            if criterio(atual.dado):
                if anterior is None:
                    self.inicio = atual.prox
                else:
                    anterior.prox = atual.prox
                self.tamanho -= 1
                return atual.dado
            anterior = atual
            atual = atual.prox
        return None

    def __iter__(self):  # permite usar "for x in lista"
        atual = self.inicio
        while atual is not None:
            yield atual.dado
            atual = atual.prox


class Fila:
    """Fila (FIFO) com ponteiros de início e fim. enfileirar/desenfileirar: O(1)."""
    def __init__(self):
        self.inicio = None
        self.fim = None
        self.tamanho = 0

    def enfileirar(self, dado):
        novo = No(dado)
        if self.fim is None:
            self.inicio = novo
        else:
            self.fim.prox = novo
        self.fim = novo
        self.tamanho += 1

    def desenfileirar(self):
        if self.inicio is None:
            return None
        dado = self.inicio.dado
        self.inicio = self.inicio.prox
        if self.inicio is None:
            self.fim = None
        self.tamanho -= 1
        return dado

    def primeiro(self):
        return None if self.inicio is None else self.inicio.dado

    def ultimo(self):
        return None if self.fim is None else self.fim.dado

    def vazia(self):
        return self.inicio is None

    def __iter__(self):
        atual = self.inicio
        while atual is not None:
            yield atual.dado
            atual = atual.prox


class Pilha:
    """Pilha (LIFO). empilhar/desempilhar: O(1)."""
    def __init__(self):
        self.topo = None
        self.tamanho = 0

    def empilhar(self, dado):
        novo = No(dado)
        novo.prox = self.topo
        self.topo = novo
        self.tamanho += 1

    def desempilhar(self):
        if self.topo is None:
            return None
        dado = self.topo.dado
        self.topo = self.topo.prox
        self.tamanho -= 1
        return dado

    def vazia(self):
        return self.topo is None


# teste rápido
f = Fila(); f.enfileirar("a"); f.enfileirar("b")
print("Fila OK:", f.desenfileirar() == "a")

Fila OK: True


In [6]:
class Veiculo:
    def __init__(self, placa, modelo, marca, ano):
        self.placa = placa.upper()
        self.modelo = modelo
        self.marca = marca
        self.ano = ano


class Cliente:
    def __init__(self, nome, cpf, telefone):
        self.nome = nome
        self.cpf = cpf
        self.telefone = telefone
        self.veiculos = Lista()  # COMBINAÇÃO: cada cliente contém uma Lista de veículos


class CadastroClientes:
    def __init__(self):
        self.clientes = Lista()

    def cadastrar_cliente(self, nome, cpf, telefone):
        if self.buscar_cliente(cpf) is not None:
            print("CPF já cadastrado.")
            return None
        cliente = Cliente(nome, cpf, telefone)
        self.clientes.adicionar(cliente)
        return cliente

    def buscar_cliente(self, cpf):
        return self.clientes.buscar(lambda c: c.cpf == cpf)

    def cadastrar_veiculo(self, cpf, placa, modelo, marca, ano):
        cliente = self.buscar_cliente(cpf)
        if cliente is None:
            print("Cliente não encontrado.")
            return None
        if self.buscar_por_placa(placa)[1] is not None:
            print("Placa já cadastrada.")
            return None
        veiculo = Veiculo(placa, modelo, marca, ano)
        cliente.veiculos.adicionar(veiculo)
        return veiculo

    def buscar_por_placa(self, placa):
        """Percorre a lista de clientes e, dentro de cada um, a lista de veículos. O(c * v)."""
        placa = placa.upper()
        for cliente in self.clientes:
            veiculo = cliente.veiculos.buscar(lambda v: v.placa == placa)
            if veiculo is not None:
                return cliente, veiculo
        return None, None

    def remover_veiculo(self, placa):
        cliente, veiculo = self.buscar_por_placa(placa)
        if veiculo is None:
            return False
        cliente.veiculos.remover(lambda v: v.placa == veiculo.placa)
        return True

    def remover_cliente(self, cpf):
        return self.clientes.remover(lambda c: c.cpf == cpf) is not None

    def listar(self):
        for c in self.clientes:
            print(f"{c.nome} | CPF {c.cpf} | Tel {c.telefone}")
            for v in c.veiculos:
                print(f"   - {v.placa}: {v.marca} {v.modelo} {v.ano}")


# teste rápido
cad = CadastroClientes()
cad.cadastrar_cliente("Teste", "000", "1")
cad.cadastrar_veiculo("000", "AAA0A00", "Uno", "Fiat", 2010)
print("Busca por placa OK:", cad.buscar_por_placa("aaa0a00")[0].nome == "Teste")

Busca por placa OK: True


In [7]:
from datetime import datetime


class Chegada:
    def __init__(self, placa, nome_cliente, queixa, guincho=False):
        self.placa = placa
        self.nome_cliente = nome_cliente
        self.queixa = queixa
        self.guincho = guincho
        self.data_hora = datetime.now()


class Box:
    def __init__(self, numero):
        self.numero = numero
        self.chegada = None  # None = box livre


class Atendimento:
    def __init__(self, qtd_boxes):
        self.fila_normal = Fila()
        self.fila_guincho = Fila()
        self.boxes = Lista()
        for i in range(1, qtd_boxes + 1):
            self.boxes.adicionar(Box(i))

    def registrar_chegada(self, placa, nome_cliente, queixa, guincho=False):
        chegada = Chegada(placa, nome_cliente, queixa, guincho)
        if guincho:
            self.fila_guincho.enfileirar(chegada)
        else:
            self.fila_normal.enfileirar(chegada)
        return chegada

    def _fila_da_vez(self):
        # COMBINAÇÃO Fila + Fila: guincho tem prioridade; se vazia, usa a normal
        if not self.fila_guincho.vazia():
            return self.fila_guincho
        return self.fila_normal

    def proximo(self):
        return self._fila_da_vez().primeiro()  # só consulta, não remove

    def chamar_proximo(self):
        box = self.boxes.buscar(lambda b: b.chegada is None)
        if box is None:
            print("Nenhum box livre.")
            return None, None
        chegada = self._fila_da_vez().desenfileirar()
        if chegada is None:
            print("Não há veículos na espera.")
            return None, None
        box.chegada = chegada
        return box, chegada

    def liberar_box(self, numero):
        box = self.boxes.buscar(lambda b: b.numero == numero)
        box.chegada = None
        prox = self.proximo()
        if prox is None:
            print(f"Box {numero} livre. Ninguém na espera.")
        else:
            tipo = "GUINCHO" if prox.guincho else "normal"
            print(f"Box {numero} livre. Próximo a ser atendido: {prox.placa} ({tipo})")


# teste rápido
at = Atendimento(1)
at.registrar_chegada("N1", "x", "q")
at.registrar_chegada("G1", "y", "q", guincho=True)
print("Prioridade guincho OK:", at.chamar_proximo()[1].placa == "G1")

Prioridade guincho OK: True


In [8]:
class NoComponente:
    def __init__(self, nome, codigo=None):
        self.nome = nome
        self.codigo = codigo      # só preenchido quando é uma peça
        self.estado = "OK"        # "OK" ou "Com defeito"
        self.filhos = Lista()     # qualquer quantidade de filhos

    def adicionar_filho(self, filho):
        self.filhos.adicionar(filho)
        return filho

    def exibir(self, nivel=0):
        """Pré-ordem: visita o nó e depois cada filho. O(n)."""
        codigo = f" [{self.codigo}]" if self.codigo else ""
        print("    " * nivel + f"{self.nome}{codigo} ({self.estado})")
        for filho in self.filhos:
            filho.exibir(nivel + 1)

    def buscar(self, nome):
        if self.nome == nome:
            return self
        for filho in self.filhos:
            achado = filho.buscar(nome)
            if achado is not None:
                return achado
        return None

    def marcar_defeito(self, nome):
        no = self.buscar(nome)
        if no is None:
            print(f"Componente '{nome}' não encontrado.")
            return False
        no.estado = "Com defeito"
        return True

    def defeitos(self, resultado=None):
        """Devolve (em Lista) todos os nós com defeito, em pré-ordem."""
        if resultado is None:
            resultado = Lista()
        if self.estado == "Com defeito":
            resultado.adicionar(self)
        for filho in self.filhos:
            filho.defeitos(resultado)
        return resultado


def montar_arvore_veiculo():
    veiculo = NoComponente("Veículo")

    motor = veiculo.adicionar_filho(NoComponente("Motor"))
    ignicao = motor.adicionar_filho(NoComponente("Sistema de ignição"))
    ignicao.adicionar_filho(NoComponente("Vela", "VL-01"))
    ignicao.adicionar_filho(NoComponente("Bobina", "BB-02"))
    motor.adicionar_filho(NoComponente("Correia dentada", "CD-03"))

    freios = veiculo.adicionar_filho(NoComponente("Freios"))
    freios.adicionar_filho(NoComponente("Pastilha dianteira", "PD-04"))
    freios.adicionar_filho(NoComponente("Disco dianteiro", "DD-05"))

    suspensao = veiculo.adicionar_filho(NoComponente("Suspensão"))
    suspensao.adicionar_filho(NoComponente("Amortecedor", "AM-06"))
    suspensao.adicionar_filho(NoComponente("Bandeja", "BJ-07"))
    return veiculo


# teste rápido
arv = montar_arvore_veiculo()
arv.marcar_defeito("Vela")
arv.exibir()

Veículo (OK)
    Motor (OK)
        Sistema de ignição (OK)
            Vela [VL-01] (Com defeito)
            Bobina [BB-02] (OK)
        Correia dentada [CD-03] (OK)
    Freios (OK)
        Pastilha dianteira [PD-04] (OK)
        Disco dianteiro [DD-05] (OK)
    Suspensão (OK)
        Amortecedor [AM-06] (OK)
        Bandeja [BJ-07] (OK)
